<a href="https://colab.research.google.com/github/serkaneren68/-ni_proje_guncel/blob/master/Reviews_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = 'serkaneren68'
os.environ['KAGGLE_KEY'] = '56535c63a5e74766c651a0d38c9658be'

In [ ]:
import kaggle
import os
import zipfile

# Replace with the actual dataset identifier (e.g., 'kaggle/titanic')
dataset_identifier = 'serkaneren68/hepsiburada-reviews'

# Create a directory for the dataset if it doesn't exist
dataset_name = dataset_identifier.split('/')[-1]
output_dir = f'./{dataset_name}'
os.makedirs(output_dir, exist_ok=True)

print(f"Downloading {dataset_identifier} to {output_dir}...")
kaggle.api.dataset_download_files(dataset_identifier, path=output_dir, unzip=True)

print("Download complete. Files are in the directory:")
print(os.listdir(output_dir))


Dataset URL: https://www.kaggle.com/datasets/serkaneren68/hepsiburada-reviews
Download complete. Files are in the directory:
['products.csv', 'reviews.csv']


In [ ]:
import pandas as pd

products_df = pd.read_csv('./hepsiburada-reviews/products.csv')
print("Products DataFrame:")
display(products_df.head())

Products DataFrame:


,id,url,title,first_seen_ts
0,1,https://www.hepsiburada.com/ceta-dijital-oda-t...,NaN,1761823722
1,2,https://www.hepsiburada.com/english-home-fancy...,NaN,1761823799
2,3,https://www.hepsiburada.com/bosch-46-parca-tor...,NaN,1761823834
3,4,https://www.hepsiburada.com/liberti-lg-marka-s...,NaN,1761823850
4,5,https://www.hepsiburada.com/flamme-cift-tarafl...,NaN,1761823984


In [ ]:
reviews_df = pd.read_csv('./hepsiburada-reviews/reviews.csv')
print("Reviews DataFrame:")
display(reviews_df.head())

Reviews DataFrame:


,id,product_id,review_hash,review_text,rating,page_no,collected_ts
0,1,1,6eac57e7345b5282f3f7eb0577e7e3321d8a665cdb961b...,Gelen ürün sipariş ettiğim ürün değil,1,1,1761823725
1,2,1,71830d5d0384a33cc95cf765b4b4fe2d2f25b7b797f074...,İçerisinde pil var diye alıyorum. Nasıl olsa p...,1,1,1761823725
2,3,1,7fbdd7702ac3d25149960478e6f9d1d31dd1f29ed3b553...,ADJ tuşu çalışmıyor bu sebepten dolayı saatini...,1,1,1761823725
3,4,1,fdee273f9a55d2638d23886aecc639adc1d7e574a8cc2a...,Hangisi doğru şimdi sıcaklıklardan ziyade nem ...,1,1,1761823725
4,5,1,e404ef5f9f39113dfd519c14d55abaa9d56a18ebd363ad...,Ekranda renk tonu bazen kayboluyor. 6 saattir ...,1,1,1761823725


In [ ]:
import pandas as pd
import re

# 1. Reload the reviews.csv file into the reviews_df DataFrame
reviews_df = pd.read_csv('./hepsiburada-reviews/reviews.csv')

# 2. Create a new column named 'original_review_text' in reviews_df
reviews_df['original_review_text'] = reviews_df['review_text'].copy()

# 3. Fill any missing values in the 'review_text' column with empty strings.
reviews_df['review_text'] = reviews_df['review_text'].fillna('')

# 4. Convert all text in the 'review_text' column to lowercase.
reviews_df['review_text'] = reviews_df['review_text'].str.lower()

# 5. Remove special characters from the 'review_text' column, preserving Turkish characters and numbers.
# Regex for Turkish lowercase alphabetic characters, numbers, and space
turkish_alpha_numeric_pattern = re.compile(r'[^a-z0-9çğıöşü\s]')
reviews_df['review_text'] = reviews_df['review_text'].apply(lambda x: turkish_alpha_numeric_pattern.sub(' ', x))

# 6. Normalize any extra spaces
reviews_df['review_text'] = reviews_df['review_text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

# 7. Display the first few entries of both the 'original_review_text' and the preprocessed 'review_text' columns.
print("Preprocessed 'review_text' column with original for comparison (first 5 entries):")
display(reviews_df[['original_review_text', 'review_text']].head())

Preprocessed 'review_text' column with original for comparison (first 5 entries):


,original_review_text,review_text
0,Gelen ürün sipariş ettiğim ürün değil,gelen ürün sipariş ettiğim ürün değil
1,İçerisinde pil var diye alıyorum. Nasıl olsa p...,i çerisinde pil var diye alıyorum nasıl olsa p...
2,ADJ tuşu çalışmıyor bu sebepten dolayı saatini...,adj tuşu çalışmıyor bu sebepten dolayı saatini...
3,Hangisi doğru şimdi sıcaklıklardan ziyade nem ...,hangisi doğru şimdi sıcaklıklardan ziyade nem ...
4,Ekranda renk tonu bazen kayboluyor. 6 saattir ...,ekranda renk tonu bazen kayboluyor 6 saattir a...


In [ ]:
duplicate_reviews = reviews_df[reviews_df['review_text'].duplicated(keep=False)]
print("Duplicate 'review_text' entries and their rows:")
display(duplicate_reviews[['review_text', 'cluster']].sort_values(by='review_text'))

Duplicate 'review_text' entries and their rows:


KeyError: "['cluster'] not in index"

In [ ]:
print(f"Reviews DataFrame before dropping duplicates: {len(reviews_df)} rows")

# Remove duplicate review_text entries, keeping the first occurrence
reviews_df.drop_duplicates(subset=['review_text'], keep='first', inplace=True)

print(f"Reviews DataFrame after dropping duplicates: {len(reviews_df)} rows")

display(reviews_df.head())

## BertTürk Modelini ve Tokenizer'ı Yükleme


In [ ]:
from transformers import AutoTokenizer, AutoModel

# 2. 'dbmdz/bert-base-turkish-cased' modelini kullanarak bir 'AutoTokenizer' nesnesi oluşturun ve bunu `tokenizer` adlı bir değişkene atayın.
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

# 3. Aynı model adını kullanarak bir 'AutoModel' nesnesi oluşturun ve bunu `model` adlı bir değişkene atayın.
model = AutoModel.from_pretrained("dbmdz/bert-base-turkish-cased")

print("BertTürk tokenizer and model loaded successfully.")

In [ ]:
import torch

# 0. Cihaz seçimi (GPU varsa onu kullan)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model.to(device)
model.eval()  # sadece embedding çıkarıyoruz, training yok

texts = reviews_df["review_text"].tolist()

batch_size = 16  # 8 / 16 / 32 deneyebilirsin, OOM olursa küçült
all_cls_embeddings = []

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i + batch_size]

    # 1. Tokenize (bu sefer her batch için)
    encoded_inputs = tokenizer(
        batch_texts,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
        max_length=512  # 512 yerine 128 de çoğu yorum için yeterli, belleği çok düşürür
    )

    # 2. Inputları cihaza gönder
    encoded_inputs = {k: v.to(device) for k, v in encoded_inputs.items()}

    # 3. Modelden geçir (grad yok)
    with torch.no_grad():
        outputs = model(**encoded_inputs)

    # 4. CLS embedding'leri al
    cls_embeddings = outputs.last_hidden_state[:, 0, :]  # (batch_size, hidden_size)

    # 5. CPU’ya çek, listeye ekle
    all_cls_embeddings.append(cls_embeddings.cpu())

# 6. Bütün batch'leri birleştir
all_cls_embeddings = torch.cat(all_cls_embeddings, dim=0)  # (num_reviews, hidden_size)

# 7. NumPy'e çevir
review_embeddings_array = all_cls_embeddings.numpy()

# 8. DataFrame'e ekle
reviews_df["review_embedding"] = list(review_embeddings_array)

# 9. Kontrol
print("reviews_df ile birlikte 'review_embedding' sütununun ilk 5 satırı:")
display(reviews_df[["review_text", "review_embedding"]].head())

print(f"Oluşturulan gömülü vektörlerin boyutu (şekli): {review_embeddings_array.shape}")


Using device: cuda
reviews_df ile birlikte 'review_embedding' sütununun ilk 5 satırı:


,review_text,review_embedding
0,gelen ürün sipariş ettiğim ürün değil,"[-0.5065468, 0.93469363, -0.060445134, 0.40317..."
1,i çerisinde pil var diye alıyorum nasıl olsa p...,"[0.88067865, -0.66265124, 0.12122536, 0.068495..."
2,adj tuşu çalışmıyor bu sebepten dolayı saatini...,"[0.74311656, -0.24477512, -0.021576121, 0.2426..."
3,hangisi doğru şimdi sıcaklıklardan ziyade nem ...,"[0.69054955, -0.14633967, -0.37960708, 0.18336..."
4,ekranda renk tonu bazen kayboluyor 6 saattir a...,"[0.7471253, 0.3193159, -0.015803551, -0.443409..."


Oluşturulan gömülü vektörlerin boyutu (şekli): (47626, 768)


## K-Ortalamalar Kümelemesi Uygulama

### Subtask:
Oluşturulan gömülü vektörlere `n_clusters=20` ile K-Ortalamalar (K-Means) kümeleme algoritmasını uygulama.


**Reasoning**:
I will import the KMeans class, instantiate it with n_clusters=20, and then fit it to the review embeddings array to perform K-Means clustering as per the instructions.



In [ ]:
from sklearn.cluster import KMeans

# 2. n_clusters parametresini 20 olarak ayarlayarak bir KMeans nesnesi oluşturun ve bunu `kmeans_model` adında bir değişkene atayın.
kmeans_model = KMeans(n_clusters=5, random_state=42, n_init=10) # n_init for modern sklearn versions

# 3. kmeans_model nesnesini, daha önce oluşturulmuş olan review_embeddings_array NumPy dizisine .fit() metodunu kullanarak uygulayın.
kmeans_model.fit(review_embeddings_array)

print("K-Means modeli başarıyla eğitildi.")

In [ ]:
labels = kmeans_model.labels_

In [ ]:
labels

In [ ]:
reviews_df["cluster"] = labels
reviews_df[["review_text", "cluster"]].head()

In [ ]:
print(reviews_df["cluster"].value_counts().sort_index())


In [ ]:
cluster_id = 0
display(reviews_df[reviews_df["cluster"] == cluster_id].head(10))


In [ ]:
centroids = kmeans_model.cluster_centers_
print(centroids.shape)   # (20, embedding_dim)

In [ ]:
import numpy as np

cluster_id = 0

# Küme merkezini al
center = centroids[cluster_id]

# Bu kümeye ait yorumların embedding’lerini al
cluster_embeddings = review_embeddings_array[reviews_df["cluster"] == cluster_id]

# Merkeze uzaklıkları hesapla (L2 distance)
distances = np.linalg.norm(cluster_embeddings - center, axis=1)

# En yakın 10 taneyi al
closest_indices = distances.argsort()[:10]

# Orijinal DataFrame index'leri
cluster_df = reviews_df[reviews_df["cluster"] == cluster_id]
representative_samples = cluster_df.iloc[closest_indices]

display(representative_samples[["review_text", "cluster"]])


In [ ]:
for i in range(5):
    subset = reviews_df[reviews_df["cluster"] == i]
    print(f"\n--- Cluster {i} ({len(subset)} reviews) ---")
    print(subset["review_text"].head(3).tolist())